# AI Solution Architecture — Reference Patterns — Hands-On

**AI Architecture · Week 19a**

Offline notebook: a compact architecture catalog, machine-readable ADRs, a hexagonal RAG pipeline, and failure injection for provider fallback. No network calls.

## 0. Reference architecture catalog

```mermaid
flowchart LR
  LLM[Simple LLM call] --> RAG[RAG with citations]
  RAG --> AGENT[Agent with tool registry]
  RAG --> EVAL[Evaluation in the loop]
  AGENT --> HITL[Human approval]
  FT[Fine-tuned serving] --> HYBRID[Hybrid RAG + fine-tune + agent]
  EVAL --> HYBRID
```

Use the smallest pattern that satisfies grounding, action, risk, and improvement requirements.

## 1. Imports

In [ ]:
from enum import Enum
from typing import List, Protocol
import numpy as np
from pydantic import BaseModel, Field

## 2. ADRs as first-class architecture data

In [ ]:
class ADRStatus(str, Enum):
    proposed = 'proposed'
    accepted = 'accepted'
    superseded = 'superseded'
    deprecated = 'deprecated'

class ADR(BaseModel):
    id: str = Field(pattern=r'^ADR-\d{4}$')
    title: str
    context: str
    decision: str
    consequences: List[str]
    alternatives: List[str]
    status: ADRStatus = ADRStatus.proposed
    def markdown(self):
        lines = [f'# {self.id} {self.title}', f'Status: {self.status.value}', '## Context', self.context, '## Decision', self.decision, '## Consequences']
        lines += [f'- {c}' for c in self.consequences]
        lines += ['## Alternatives'] + [f'- {a}' for a in self.alternatives]
        return '\n'.join(lines)

class ADRRegistry:
    def __init__(self): self.records = {}
    def add(self, adr):
        if adr.id in self.records: raise ValueError('duplicate ADR')
        self.records[adr.id] = adr
    def accepted(self): return [a for a in self.records.values() if a.status == ADRStatus.accepted]

registry = ADRRegistry()
for adr in [
    ADR(id='ADR-0001', title='Choose pgvector over Pinecone for MVP', context='Small corpus, Postgres ops maturity, tenant joins.', decision='Use pgvector for release 1.', consequences=['Simple ops', 'Revisit at 5M chunks or p95 miss'], alternatives=['Pinecone', 'Qdrant'], status=ADRStatus.accepted),
    ADR(id='ADR-0002', title='Adopt hexagonal provider ports', context='Tenant procurement may require Azure OpenAI, Bedrock, or Vertex.', decision='Use LLMProvider and EmbeddingProvider ports.', consequences=['Swappable providers', 'Adapter tests required'], alternatives=['Direct SDK imports'], status=ADRStatus.accepted),
    ADR(id='ADR-0003', title='Choose RAG over fine-tune for policy assistant', context='Policies change weekly and answers need citations.', decision='Use RAG with golden-query evals.', consequences=['Grounded answers', 'Retrieval quality becomes a release gate'], alternatives=['Fine-tune only', 'Long context only'], status=ADRStatus.accepted),
]: registry.add(adr)
print([a.id for a in registry.accepted()])
print(registry.records['ADR-0003'].markdown().splitlines()[:4])

## 3. Hexagonal RAG pipeline: ports

The application depends on protocols, not SDKs. Azure OpenAI, Bedrock, Vertex, pgvector, Pinecone, Qdrant, or test fakes are adapters.

In [ ]:
class LLMProvider(Protocol):
    def complete(self, prompt: str) -> str: ...
class EmbeddingProvider(Protocol):
    def embed(self, text: str) -> np.ndarray: ...
class VectorStore(Protocol):
    def add(self, doc_id: str, text: str, vector: np.ndarray): ...
    def search(self, vector: np.ndarray, k: int = 3) -> list[dict]: ...
class Reranker(Protocol):
    def rerank(self, query: str, docs: list[dict]) -> list[dict]: ...
class Guardrail(Protocol):
    def check(self, text: str) -> str: ...
class EvalRecorder(Protocol):
    def record(self, event: dict): ...

## 4. Fake adapters and RAGPipeline

In [ ]:
class FakeEmbedder:
    def embed(self, text):
        words = set(text.lower().split())
        return np.array([len(words), int('refund' in words), int('policy' in words), int('mfa' in words)], dtype=float)

class MemoryVectorStore:
    def __init__(self): self.rows = []
    def add(self, doc_id, text, vector): self.rows.append({'id': doc_id, 'text': text, 'vector': vector})
    def search(self, vector, k=3):
        def score(row):
            denom = np.linalg.norm(vector) * np.linalg.norm(row['vector']) or 1.0
            return float(np.dot(vector, row['vector']) / denom)
        return sorted(({**r, 'score': score(r)} for r in self.rows), key=lambda r: r['score'], reverse=True)[:k]

class KeywordReranker:
    def rerank(self, query, docs):
        terms = set(query.lower().split())
        return sorted(docs, key=lambda d: len(terms & set(d['text'].lower().split())), reverse=True)

class SimpleGuardrail:
    def check(self, text):
        if 'ignore previous instructions' in text.lower():
            return 'REFUSE: prompt injection detected'
        return text

class FakeLLM:
    def complete(self, prompt):
        return 'Refunds are allowed within 30 days with receipt. [doc:refund-policy]'

class ListEvalRecorder:
    def __init__(self): self.events = []
    def record(self, event): self.events.append(event)

class RAGPipeline:
    def __init__(self, llm, embedder, store, reranker, guardrail, evals):
        self.llm=llm; self.embedder=embedder; self.store=store; self.reranker=reranker; self.guardrail=guardrail; self.evals=evals
    def ingest(self, doc_id, text):
        clean = self.guardrail.check(text)
        self.store.add(doc_id, clean, self.embedder.embed(clean))
        self.evals.record({'stage': 'ingest', 'doc_id': doc_id})
    def answer(self, query):
        checked = self.guardrail.check(query)
        docs = self.reranker.rerank(checked, self.store.search(self.embedder.embed(checked)))
        context = '\n'.join(f"[{d['id']}] {d['text']}" for d in docs)
        answer = self.llm.complete(f'Use only cited context.\n{context}\nQuestion: {checked}')
        self.evals.record({'stage': 'answer', 'docs': [d['id'] for d in docs]})
        return answer

recorder = ListEvalRecorder()
pipeline = RAGPipeline(FakeLLM(), FakeEmbedder(), MemoryVectorStore(), KeywordReranker(), SimpleGuardrail(), recorder)
pipeline.ingest('doc:refund-policy', 'Refund policy allows refunds within 30 days with receipt.')
pipeline.ingest('doc:mfa-policy', 'Security policy requires MFA for administrator access.')
print(pipeline.answer('What is the refund policy?'))
print(recorder.events)

## 5. Failure injection: primary provider outage with fallback

In [ ]:
class FailingLLM:
    def complete(self, prompt):
        raise TimeoutError('primary provider outage')

class FallbackRouter:
    def __init__(self, primary, fallback, recorder):
        self.primary = primary; self.fallback = fallback; self.recorder = recorder
    def complete(self, prompt):
        try:
            return self.primary.complete(prompt)
        except Exception as exc:
            self.recorder.record({'stage': 'llm_fallback', 'error': type(exc).__name__})
            return self.fallback.complete(prompt)

failure_recorder = ListEvalRecorder()
resilient = RAGPipeline(FallbackRouter(FailingLLM(), FakeLLM(), failure_recorder), FakeEmbedder(), MemoryVectorStore(), KeywordReranker(), SimpleGuardrail(), failure_recorder)
resilient.ingest('doc:refund-policy', 'Refund policy allows refunds within 30 days with receipt.')
print(resilient.answer('What is the refund policy?'))
print(failure_recorder.events)

## Exercises
1. Add a `CostLedger` port and record estimated token cost per answer.
2. Add a canary vector index and compare golden-query recall before promotion.
3. Add a `HumanApprovalQueue` adapter for tool calls above a risk threshold.
4. Write an ADR superseding pgvector with Qdrant and include the revisit trigger.

## Links
- Literature note: `02 Literature Notes/AI Architecture/AI Solution Architecture — Reference Patterns`
- Snippets: `04 Code Snippets/AI Architecture/AI Week 19a Machine Readable ADR Registry`, `.../AI Week 19a Hexagonal RAG Pipeline Demo`
- MOC: `06 Maps of Content/AI Architecture Concepts`